In [1]:
%load_ext autoreload
%autoreload 2

# Summary

Collect logprobs for joke dataset. Would have been nice to do this upfront but used gpt-5-mini. Regardless, it would be nice to have this functionality in general.

In [2]:
import gc
import os
from pathlib import Path
from typing import Optional, Union, Any

In [3]:
repo_parent = Path(".").absolute().parent.parent
os.environ["HF_HOME"] = str(repo_parent/".cache")

In [4]:
# Think we need to import after setting env var if we want custom cache dir.
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

from tqdm.auto import tqdm
from datasets import Dataset, load_dataset
import pandas as pd
import numpy as np
from huggingface_hub import login, HfApi
from pyperclip import copy, paste

from aeon.secrets import SecretManager
from aeon import config
from aeon import datasets
from aeon.utils import timer

In [5]:
name = "Qwen/Qwen3-8B-Base"
tokenizer = AutoTokenizer.from_pretrained(name)
model = AutoModelForCausalLM.from_pretrained(name).to("cuda")

Loading checkpoint shards:   0%|          | 0/5 [00:00<?, ?it/s]

In [89]:
def get_text_logprobs(prompt: Optional[str], text: str, model, tokenizer, k: int = 10, i2w: Optional[dict] = None):
    """For an existing text sequence (can be LLM-generated, human-written, whatever),
    get some model's logprobs for each token. Essentially shows us how surprising each
    token was.
    
    Parameters
    ----------
    k : int
        Number of most probably tokens to return logprobs for at each step.
    i2w : dict or NoneType
        Maps tokenizer token index (int) to token (str). If not provided, we will
        construct it from the `tokenizer` arg. (Just saves a little time to not have
        to iterate over the whole vocab an extra time on every batch since we want to
        run this func on any inputs.)
    """
    model.generation_config.pad_token_id = tokenizer.pad_token_id
    vocab = tokenizer.get_vocab()
    i2w = i2w or {i: word for word, i in tokenizer.get_vocab().items()}
    tokens = tokenizer.tokenize(text)
    # We will use these as labels later.
    token_idx = torch.tensor([vocab[t] for t in tokens], device=model.device)
    # list[str]
    sequences = [
        tokenizer.convert_tokens_to_string(tokens[:i])
        for i in np.arange(len(tokens))
    ]

    all_inputs = []
    base_messages = [{"role": "system", "content": prompt}] if prompt else []
    for seq in sequences:
        messages = base_messages + [{"role": "user", "content": seq}]
        inputs = tokenizer.apply_chat_template(
        	messages,
            continue_final_message=True,
        	add_generation_prompt=False,
        	tokenize=True,
        	return_dict=True,
        	return_tensors="pt",
        )
        inputs["input_ids"] = inputs["input_ids"].squeeze()
        inputs["attention_mask"] = inputs["attention_mask"].squeeze()
        all_inputs.append(inputs)

    padded_inputs = tokenizer.pad(all_inputs, padding=True, padding_side="left").to(model.device)
    
    outputs = model.generate(**padded_inputs, max_new_tokens=1,
                             return_dict_in_generate=True, output_scores=True)

    # outputs.scores has len max_new_tokens which is always 1 in our case.
    # Just pull out the relevant bit for easy handling.
    # shape: (bs, vocab_size)
    scores = outputs.scores[0]
    logprobs_allrows = scores.log_softmax(dim=-1)
    # logprob for correct next token for each row.
    label_logprobs = logprobs_allrows[torch.arange(logprobs_allrows.shape[0]).to(model.device), token_idx]
    
    # Get index of top 10 logprobs for each row
    idx_allrows = logprobs_allrows.argsort(dim=-1, descending=True)
    label_rank = (idx_allrows == token_idx.unsqueeze(-1)).nonzero()[:, -1]
    idx_topk = idx_allrows[:, :k]
    logprobs_topk = logprobs_allrows.gather(-1, idx_topk)

    res = []
    for label, label_logprob, rank, idx, logprobs in zip(
        tokens, label_logprobs, label_rank, idx_topk, logprobs_topk
    ):
        probs = logprobs.exp()
        item = {
            "label": label,
            "label_prob": label_logprob.exp().item(),
            "label_rank": rank.item(),
            "label_logprob": label_logprob.item(),
            "top_k_probs": {
                i2w[i.item()]: prob.item() for i, prob in zip(idx, probs)
            },
            "top_k_logprobs": {
                i2w[i.item()]: logprob.item() for i, logprob in zip(idx, logprobs)
            },
        }
        res.append(item)
    return res

In [86]:
vocab = tokenizer.get_vocab()
i2w = {i: word for word, i in vocab.items()}

In [87]:
res = get_text_logprobs("Tell me about the sky.", "The sky is blue and grass is green.", model, tokenizer, i2w=i2w)

In [28]:
pd.DataFrame(res)

,label,label_prob,label_rank,label_logprob,top_k_probs,top_k_logprobs
0,The,0.171236,0,-1.764715,"{'The': 0.1712355613708496, 'Tell': 0.14073315...","{'The': -1.7647150754928589, 'Tell': -1.960889..."
1,Ġsky,0.933374,0,-0.068950,"{'Ġsky': 0.9333735108375549, 'ĠSky': 0.0103762...","{'Ġsky': -0.06894978135824203, 'ĠSky': -4.5682..."
2,Ġis,0.705182,0,-0.349300,"{'Ġis': 0.7051815986633301, ',': 0.10386520624...","{'Ġis': -0.34929996728897095, ',': -2.26466131..."
3,Ġblue,0.046402,2,-3.070406,"{'Ġa': 0.4470357596874237, 'Ġthe': 0.230530753...","{'Ġa': -0.8051167130470276, 'Ġthe': -1.4673709..."
4,Ġand,0.097452,2,-2.328398,"{'.': 0.23665662109851837, '.Ċ': 0.19150346517...","{'.': -1.4411450624465942, '.Ċ': -1.6528493165..."
5,Ġgrass,0.000010,821,-11.465742,"{'Ġwhite': 0.13064095377922058, 'Ġthe': 0.1048...","{'Ġwhite': -2.0353026390075684, 'Ġthe': -2.255..."
6,Ġis,0.887614,0,-0.119219,"{'Ġis': 0.887613832950592, 'y': 0.066946402192...","{'Ġis': -0.11921855807304382, 'y': -2.70386290..."
7,Ġgreen,0.951815,0,-0.049384,"{'Ġgreen': 0.9518154859542847, 'Ġblue': 0.0056...","{'Ġgreen': -0.04938405752182007, 'Ġblue': -5.1..."
8,.,0.291438,1,-1.232930,"{'.Ċ': 0.32165002822875977, '.': 0.29143750667...","{'.Ċ': -1.134291172027588, '.': -1.23292970657..."


In [5]:
ds = load_dataset("hmamin/extract_jokes")

In [6]:
df = ds['train'].to_pandas()

In [8]:
df.tail(2)

,prompt,joke,subtext,unfunny_variant,web_scraper_order,transcript_link,transcript_link_href
22927,Do you remember things you said on drugs?,"I gave an interview to GQ December 15th, 2020 ...",Substance use impairs memory and leads to inco...,I don't remember certain interviews I apparent...,1686242983-419,John Mulaney: Baby J (2023) | Transcript,https://scrapsfromtheloft.com/comedy/john-mula...
22928,"If you had a talk show, what would it be like?","GQ asked if I'd want my own talk show. I said,...",Some talk show concepts are oddly specific or ...,"I once thought about two talk show concepts, i...",1686242983-419,John Mulaney: Baby J (2023) | Transcript,https://scrapsfromtheloft.com/comedy/john-mula...


In [31]:
row = df.sample(1)

In [32]:
res = get_text_logprobs(row.prompt.values[0], row.joke.values[0], model, tokenizer, i2w=i2w)

Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


In [33]:
pd.DataFrame(res)

,label,label_prob,label_rank,label_logprob,top_k_probs,top_k_logprobs
0,You,0.007833,17,-4.849458,"{'Yes': 0.08825743198394775, 'I': 0.0826682671...","{'Yes': -2.427497386932373, 'I': -2.4929194450..."
1,âĢĻve,0.003204,38,-5.743373,"{''re': 0.1151023581624031, 'Ġare': 0.11087415...","{''re': -2.161933422088623, 'Ġare': -2.1993594..."
2,Ġnever,0.020481,6,-3.888256,"{'Ġgot': 0.1725987195968628, 'Ġprobably': 0.13...","{'Ġgot': -1.7567859888076782, 'Ġprobably': -1...."
3,Ġseen,0.332288,0,-1.101754,"{'Ġseen': 0.33228763937950134, 'Ġbeen': 0.1000...","{'Ġseen': -1.1017543077468872, 'Ġbeen': -2.301..."
4,Ġa,0.330201,0,-1.108055,"{'Ġa': 0.3302007019519806, 'Ġanything': 0.0814...","{'Ġa': -1.1080546379089355, 'Ġanything': -2.50..."
5,Ġfucking,0.000131,161,-8.937785,"{'Ġghost': 0.8041595816612244, 'Ġreal': 0.0216...","{'Ġghost': -0.21795758605003357, 'Ġreal': -3.8..."
6,Ġghost,0.873207,0,-0.135582,"{'Ġghost': 0.8732073903083801, 'Ġthing': 0.011...","{'Ġghost': -0.13558222353458405, 'Ġthing': -4...."
7,.,0.188246,0,-1.670005,"{'.': 0.18824611604213715, ',': 0.164621725678...","{'.': -1.6700050830841064, ',': -1.80410504341..."
8,ĠNot,0.008287,21,-4.793097,"{'ĠYou': 0.117436483502388, 'ĠI': 0.0466322675...","{'ĠYou': -2.141857624053955, 'ĠI': -3.06546258..."
9,Ġone,0.181056,1,-1.708951,"{'Ġeven': 0.22972415387630463, 'Ġone': 0.18105...","{'Ġeven': -1.4708760976791382, 'Ġone': -1.7089..."


## Run on full dataset

In [6]:
out_dir = config.DATA_DIR/'tmp'
dataset_dir = config.DATA_DIR/"datasets/extract_jokes_logprobs"
os.makedirs(dataset_dir, exist_ok=True)

In [ ]:
all_res = []
for i, row in tqdm(df.iterrows()):
    try:
        res = get_text_logprobs(row.prompt, row.joke, model, tokenizer, i2w=i2w)
        df_res = pd.DataFrame(res)
        df_res.to_parquet(out_dir/f"{i}.pq")
    except Exception as e:
        print(f'[row {i}] error: {e}')
    else:
        all_res.append(df_res)

    # Progress bar rendering is flaky, add a backup way to monitor progress.
    # In practice `watch`ing the data/tmp dir is probably better though.
    if not i % 100:
        print(i)

In [16]:
def drop_nones(tok2p: dict):
    """
    Noticed weird bug after running where the probs/logprobs cols seemed to store maybe extra keys
    with value None. Guessing (?) maybe this is the union of all the rows' logprobs dicts' keys?
    Regardless, the non-None vals and corresponding keys *seem* to be correct so let's just filter
    out the Nones. Can investigate root cause later so this isn't necessary. For now we should apply
    this to both the top_k_probs and top_k_logprobs cols.
    
    """
    return dict(
        sorted(
            {k: v for k, v in tok2p.items() if v is not None}.items(),
            reverse=True, key=lambda x: x[1]
        )
    )

In [11]:
i2df = {}
for path in tqdm(out_dir.iterdir(), total=df.shape[0]):
    if path.suffix != ".pq":
        continue
    i2df[int(path.stem)] = pd.read_parquet(path)

  0%|          | 0/22929 [00:00<?, ?it/s]

In [13]:
# Used this on the initial run when all the tiny dfs were already in memory. Had to adjust logic a bit
# on subsequent working sessions since they're no longer in memory.
# df_all = pd.concat([dfi.assign(id=i) for i, dfi in enumerate(all_res)], axis=0).reset_index(drop=True)

# Updated logic after loading from disk into a dict (instead of list like we used on initial run).
df_all = pd.concat([dfi.assign(id=i) for i, dfi in i2df.items()], axis=0).reset_index(drop=True)

In [ ]:
df_all["top_k_probs"] = df_all['top_k_probs'].apply(drop_nones)
df_all['top_k_logprobs'] = df_all['top_k_logprobs'].apply(drop_nones)

In [32]:
df_all = df_all.reset_index().sort_values(['id', 'index']).reset_index(drop=True)

In [38]:
df_grouped = df_all.groupby('id')[df_all.drop(columns=["id", "index"]).columns.tolist()].agg(list)

In [41]:
df_grouped = df_grouped.reset_index()

In [ ]:
tmp = pd.merge(df, df_grouped, how='inner', left_index=True, right_on='id').drop(columns="id")
# This turns out to be a huge df that takes ages to save so let's remove some duplicate data.
# Not hard to recover these from the probs, which mostly aren't that small anyway in this case
# because we chose the top k.
tmp = tmp.drop(columns=["top_k_logprobs", "label_logprob"])

In [70]:
with timer():
    tmp.to_hdf(dataset_dir/"df.h5", key="df")

/tmp/ipykernel_1965/1176776786.py:2: PerformanceWarning: 
your performance may suffer as PyTables will pickle object types that it cannot
map directly to c-types [inferred_type->mixed,key->block0_values] [items->Index(['prompt', 'joke', 'subtext', 'unfunny_variant', 'web_scraper_order',
       'transcript_link', 'transcript_link_href', 'label', 'label_prob',
       'label_rank', 'top_k_probs'],
      dtype='object')]

  tmp.to_hdf(dataset_dir/"df_logprobs.h5", key="df")
[TIMER] BLOCK executed in 3.491 s.


In [7]:
tmp = pd.read_hdf(dataset_dir/"df.h5", key="df")

In [8]:
tmp['uuid'] = tmp.web_scraper_order + '-' + tmp.index.astype(str)

In [9]:
# trouble connecting to infisical so huggingface save fails
datasets.save_dataset(
    tmp, "extract_jokes_logprobs", save_local=False,
)#, file_suffix="h5")

ProxyError: HTTPSConnectionPool(host='app.infisical.com', port=443): Max retries exceeded with url: /api/v1/auth/universal-auth/login (Caused by ProxyError('Unable to connect to proxy', OSError('Tunnel connection failed: 403 Forbidden')))